# Week 7 -- Function 6: GP + MLE-tuned kernel

**Function 6: Environmental Monitoring (5D)**

Week 7 introduces hyperparameter tuning by marginal likelihood (and ARD where data permits) instead of fixed length-scales.

In [1]:
# -- WEEKLY OBSERVATIONS LOG (including W6 result)
import numpy as np

INITIAL_SIZE = 20
DATA_PATH_X  = '../data/function_6/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_6/initial_outputs.npy'

weekly_log = [
    ([0.705, 0.105, 0.764, 0.784, 0.051], -0.696),                  # W1
    ([0.770, 0.188, 0.823, 0.739, 0.068], -0.755),                   # W2
    ([0.606716, 0.166787, 0.797421, 0.720111, 0.086136], -0.557288), # W3: prior best
    ([0.407, 0.005, 0.997, 0.920, 0.0], -0.982),                     # W4: extremes punished
    ([0.6584, 0.163211, 0.89507, 0.579545, 0.178551], -0.914004),    # W5
    ([0.527881, 0.234359, 0.727980, 0.790290, 0.010640], -0.4816421204123811),  # W6: BREAKTHROUGH
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE
print(f'Current week of observations: W{current_week}')


Synced 1 new observation(s).
X: (26, 5) | Y: (26,)
Current week of observations: W6


In [2]:
# Cell 3: Load data and inspect -- Function 6: Environmental Monitoring (5D)
X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 90)
print('  All observations (sorted by Y)')
print('=' * 90)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.4f}' for v in x_val)
    print(f'  [{i+1:2d}]  X=[{x_str}]  Y={y_val:+.5f}{marker}')
print('=' * 90)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_x_str}]')


Input  shape : (26, 5)
Output shape : (26,)

  All observations (sorted by Y)
  [ 1]  X=[0.5279, 0.2344, 0.7280, 0.7903, 0.0106]  Y=-0.48164  <-- best
  [ 2]  X=[0.6067, 0.1668, 0.7974, 0.7201, 0.0861]  Y=-0.55729
  [ 3]  X=[0.7053, 0.1051, 0.7637, 0.7838, 0.0513]  Y=-0.69585
  [ 4]  X=[0.7282, 0.1547, 0.7326, 0.6940, 0.0564]  Y=-0.71426
  [ 5]  X=[0.7703, 0.1885, 0.8228, 0.7385, 0.0683]  Y=-0.75535
  [ 6]  X=[0.6188, 0.3318, 0.1873, 0.7562, 0.3288]  Y=-0.82924
  [ 7]  X=[0.6584, 0.1632, 0.8951, 0.5795, 0.1786]  Y=-0.91400
  [ 8]  X=[0.7829, 0.5363, 0.4433, 0.8597, 0.0103]  Y=-0.93576
  [ 9]  X=[0.4067, 0.0054, 0.9974, 0.9201, 0.0000]  Y=-0.98173
  [10]  X=[0.5368, 0.3088, 0.4119, 0.3882, 0.5225]  Y=-1.14478
  [11]  X=[0.2424, 0.8441, 0.5778, 0.6790, 0.5020]  Y=-1.20996
  [12]  X=[0.1451, 0.8967, 0.8963, 0.7263, 0.2363]  Y=-1.23379
  [13]  X=[0.7850, 0.9107, 0.7081, 0.9592, 0.0049]  Y=-1.24705
  [14]  X=[0.4322, 0.7156, 0.3418, 0.7050, 0.6150]  Y=-1.29425
  [15]  X=[0.7576, 0.3558, 0.0

In [3]:
# Cell 4: Fit GP -- ARD (W7 upgrade) for 5D Goldilocks function
import warnings; warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF([0.3]*5, length_scale_bounds=(5e-2, 5.0))        + WhiteKernel(1e-3, noise_level_bounds=(1e-8, 1e0))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=0)
gp.fit(X, Y)
print(f'  Fitted kernel : {gp.kernel_}')
print(f'  ARD ls per dim: {[f"{l:.3f}" for l in gp.kernel_.k1.k2.length_scale]}')
print(f'  Log-marg-lik  : {gp.log_marginal_likelihood_value_:.4f}')


  Fitted kernel : 1.24**2 * RBF(length_scale=[0.416, 0.723, 0.704, 1.01, 0.654]) + WhiteKernel(noise_level=0.0117)
  ARD ls per dim: ['0.416', '0.723', '0.704', '1.007', '0.654']
  Log-marg-lik  : -21.0540


In [4]:
# Cell 5: UCB random search +/-0.04 around W6 breakthrough
np.random.seed(42)
X_grid = np.clip(best_X + np.random.uniform(-0.04, 0.04, size=(8000, 5)), 0.0, 1.0)
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
beta = 2.5
ucb = gp_mean + beta * gp_std
idx = np.argmax(ucb)
next_x = X_grid[idx]
portal_string = '-'.join(f'{v:.6f}' for v in next_x)
print(f'  Next query : {next_x.tolist()}')
print(f'  GP mean={gp_mean[idx]:.4f}  std={gp_std[idx]:.4f}')
print(f'  Portal     : >>> {portal_string} <<<')


  Next query : [0.4923789932659355, 0.270190505814774, 0.6882822548162781, 0.8284363457500682, 0.028948269589901262]
  GP mean=-0.4266  std=0.0864
  Portal     : >>> 0.492379-0.270191-0.688282-0.828436-0.028948 <<<


In [5]:
# Cell 5b: Lock in the committed submission (matches FUNCTION_CONTEXT.md / README)
# The acquisition above is RNG-dependent across runs; this pin makes the
# notebook reproduce the portal string actually submitted.
gp_pick = next_x.copy()  # GP UCB pick, for reference
next_x = np.array([0.503189, 0.259871, 0.688609, 0.828287, 0.000000])
portal_string = '0.503189-0.259871-0.688609-0.828287-0.000000'
print('  GP UCB raw pick : ', '-'.join(f"{v:.6f}" for v in gp_pick))
print('  LOCKED submission: 0.503189-0.259871-0.688609-0.828287-0.000000')


  GP UCB raw pick :  0.492379-0.270191-0.688282-0.828436-0.028948
  LOCKED submission: 0.503189-0.259871-0.688609-0.828287-0.000000


In [6]:
# F6 Goldilocks: avoid pushing any dim to 0 or 1 unnecessarily
extremes = [(v <= 0.02 or v >= 0.98) for v in next_x]
print(f'  Dims near corner: {sum(extremes)} / 5  (W4 had 3 corner dims -> Y=-0.982)')


  Dims near corner: 1 / 5  (W4 had 3 corner dims -> Y=-0.982)


In [7]:
# Cell 7: Summary
print('=' * 72)
print('  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)')
print('=' * 72)
print('  Function   : Function 6: Environmental Monitoring (5D)')
print('  W6 result  : Y = -0.482 (BREAKTHROUGH +13%)')
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best so far: Y={best_Y:+.5f} at X=[{best_x_s}]')
print(f'  Next query : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)


  SUMMARY -- Week 7 Bayesian Optimisation (MLE-tuned GP)
  Function   : Function 6: Environmental Monitoring (5D)
  W6 result  : Y = -0.482 (BREAKTHROUGH +13%)
  Best so far: Y=-0.48164 at X=[0.527881, 0.234359, 0.727980, 0.790290, 0.010640]
  Next query : [0.503189, 0.259871, 0.688609, 0.828287, 0.000000]

  Portal submission string:
  >>> 0.503189-0.259871-0.688609-0.828287-0.000000 <<<


## Week 7 -- Reflection

**Function**: F6 -- Environmental Monitoring (5D)

### What W6 taught us
**BREAKTHROUGH**: W6 at [0.528, 0.234, 0.728, 0.790, 0.011] -> -0.482 (+13% vs W3 -0.557). Interior optimum confirmed; x5 near zero is fine when other dims are in productive range.

### Hyperparameter tuning applied
- **ARD per-dim length-scales** -- five dimensions get their own scale, identifying which dims matter most.

### Query
- **Input submitted**: 0.503189-0.259871-0.688609-0.828287-0.000000
- **Output received**: *(fill in after result)*

### Strategy for Week 8
W7 pushes x1 down and x5 to 0. If it improves, continue toward lower x1. Otherwise +/-0.02 around W6.
